### 1. Initialze初始化

In [8]:
import os
import sys
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# 1. 挂载 Google Drive 并设置路径
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Final_Project'
IMG_DIR = f'{PROJECT_DIR}/original_data/01'
CSV_PATH = f'{PROJECT_DIR}/train.csv'

# 将项目路径加入 sys.path 以便导入自定义模块
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Country -> Continent

In [12]:
import os
import pandas as pd
from tqdm.auto import tqdm
from hash import country_to_continent

tqdm.pandas()

# 1. 获取本地图片 ID
print("1. 正在读取本地图片列表...")
with open('/content/local_images.txt', 'r') as f:
    existing_ids = set([line.strip().split('.')[0] for line in f.readlines() if line.strip().endswith(('.jpg', '.png'))])

# 2. 分块读取并过滤
print("2. 正在分块读取并过滤 500 万行 CSV...")
chunk_size = 100000
estimated_chunks = 50
filtered_chunks = []

with pd.read_csv(CSV_PATH, dtype=str, chunksize=chunk_size, low_memory=False) as reader:
    for chunk in tqdm(reader, total=estimated_chunks, desc="CSV 过滤进度"):
        # 假设 ID 是第一列，或者你可以直接用 chunk['id'] 如果列名确定是 id
        id_col = chunk.columns[0]
        chunk['id_clean'] = chunk[id_col].str.strip().str.replace('.jpg', '', regex=False).str.replace('.png', '', regex=False)

        valid_chunk = chunk[chunk['id_clean'].isin(existing_ids)].copy()

        if not valid_chunk.empty:
            # 🚨 核心修复：直接通过列名 'country' 来获取国家代码，而不是用索引！
            if 'country' in valid_chunk.columns:
                valid_chunk['country_clean'] = valid_chunk['country'].astype(str).str.strip()
            else:
                raise ValueError(f"CSV 中找不到 'country' 列！实际的列名有: {valid_chunk.columns.tolist()}")

            filtered_chunks.append(valid_chunk[['id_clean', 'country_clean']])

df_filtered = pd.concat(filtered_chunks, ignore_index=True)
print(f"\n-> CSV 扫描完成！成功匹配到 {len(df_filtered)} 条元数据。")

# 🚨 抽查一下提取出来的国家代码长什么样
print(f"-> 提取的国家代码示例 (前5个): {df_filtered['country_clean'].head(5).tolist()}")

# 3. 映射大洲标签
print("\n3. 正在将 Country 映射为 Continent...")
df_filtered['label'] = df_filtered['country_clean'].progress_apply(country_to_continent)

# 看看有多少映射失败的
failed_mapping = df_filtered['label'].isna().sum()
print(f"-> 无法映射到大洲的国家数量: {failed_mapping} (将被丢弃)")

df_filtered = df_filtered.dropna(subset=['label']).reset_index(drop=True)
df_filtered = df_filtered.rename(columns={'id_clean': 'id'})

unique_continents = sorted(df_filtered['label'].unique().tolist())
label_to_idx = {label: i for i, label in enumerate(unique_continents)}
NUM_CLASSES = len(unique_continents)

print(f"\n-> 标签映射完成！最终可用于训练的有效数据量: {len(df_filtered)}")
print(f"-> 类别数量: {NUM_CLASSES}")
print(f"-> 类别字典: {label_to_idx}")


1. 正在读取本地图片列表...
2. 正在分块读取并过滤 500 万行 CSV...


CSV 过滤进度:   0%|          | 0/50 [00:00<?, ?it/s]


-> CSV 扫描完成！成功匹配到 50000 条元数据。
-> 提取的国家代码示例 (前5个): ['NZ', 'CL', 'FK', 'CL', 'AR']

3. 正在将 Country 映射为 Continent...


  0%|          | 0/50000 [00:00<?, ?it/s]

-> 无法映射到大洲的国家数量: 285 (将被丢弃)

-> 标签映射完成！最终可用于训练的有效数据量: 49715
-> 类别数量: 10
-> 类别字典: {'East_Asia': 0, 'East_Europe_CIS': 1, 'Latin_America': 2, 'Med_MENA': 3, 'North_America': 4, 'Oceania_Islands': 5, 'South_Central_Asia': 6, 'Southeast_Asia': 7, 'Sub_Saharan_Africa': 8, 'West_North_Europe': 9}


### 3. 生成数据类

In [ ]:
class ContinentDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. 取出对应的行并获取真正的图片路径
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])

        print(f"\n[DEBUG] Dataset getitem called for idx={idx}")
        print(f"[DEBUG] Opening image: {img_path}")

        # 2. 读取图片 (如果这里卡住，控制台就不会打印下一行！)
        image = Image.open(img_path).convert('RGB')
        
        print(f"[DEBUG] Image opened successfully!")

        if self.transform:
            image = self.transform(image)

        label_idx = label_to_idx[row['label']]
        return image, torch.tensor(label_idx, dtype=torch.long)


### 用transform高效crop
如果来来回回读写，将会极其低效。

In [14]:
# 定义标准的 ResNet 预处理流水线：最短边缩放至 256 -> 中心裁剪 224x224 -> 归一化
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 实例化完整数据集
full_dataset = ContinentDataset(df, IMG_DIR, transform=transform)


### 把数据集切分成2部分

In [15]:
# 计算切分大小
test_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - test_size

# 随机切分
train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42) # 固定随机种子，保证每次切分一致
)

# 构建 DataLoader (建议 num_workers=2 或 4 以加速读取)
BATCH_SIZE = 64 # 根据你的显存大小调整
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Dataset split complete: {len(train_dataset)} Train, {len(test_dataset)} Test.")


ValueError: num_samples should be a positive integer value, but got num_samples=0

### 开始训练

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 1. 初始化模型、损失函数和优化器
model = resnet.build_resnet50(num_classes=NUM_CLASSES).to(device)
criterion = loss.build_loss()
optimizer, scheduler = resnet.build_training_components(model, learning_rate=0.001)

# 2. 开始训练 (假设跑 10 个 Epoch)
print("Starting training...")
history = train_module.train_model(
    model=model,
    train_loader=train_loader,
    val_loader=None,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=10,
    device=device
)

# 3. 在 Testset 上进行预测与评估
print("\nEvaluating on Test Set...")
test_loss, test_acc = train_module.evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device
)
print(f"Final Test Accuracy: {test_acc * 100:.2f}%")

# 4. 将训练好的模型保存到 Google Drive
save_path = os.path.join(PROJECT_DIR, 'resnet50_continent_best.pth')
torch.save(model.state_dict(), save_path)
print(f"Model successfully saved to: {save_path}")
